Fine-Tuning Transformer Models for Question Answering

1. Introduction

Question Answering (QA) tasks differ from traditional classification tasks in how predictions are made. In classification, the model assigns a fixed label (such as positive or negative) to an input. However, in QA, the model identifies a relevant span of text from a given context to answer a question. Instead of predicting labels, QA models predict the start and end positions of the answer within the context. This makes QA a span prediction problem rather than a classification problem.

2. 2. Implementation

2.1 Environment Setup

In [1]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


2.2 Load and Explore Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("squad")

# To watch
dataset["train"][0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

2.3 BERT Model Setup

In [3]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

2.4 Tokenization and Preprocessing

In [6]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs["offset_mapping"]
    answers = examples["answers"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        # Find the start and end of the context in the tokenized sequence
        sequence_ids = inputs.sequence_ids(i)
        context_start_token = None
        context_end_token = None
        for k, sid in enumerate(sequence_ids):
            if sid == 1: # Part of the context
                if context_start_token is None:
                    context_start_token = k
                context_end_token = k

        if context_start_token is None or context_end_token is None: # No context found
            start_positions.append(0)
            end_positions.append(0)
            continue

        answer = answers[i]
        # Handle cases where answer['text'] might be empty or missing
        if not answer["text"] or not answer["text"][0]:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        # If the answer is not fully contained in the context, label it as (0, 0)
        if not (offsets[context_start_token][0] <= start_char and offsets[context_end_token][1] >= end_char):
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise, find the start and end token indices of the answer
            token_start_index = context_start_token
            while token_start_index <= context_end_token and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            token_end_index = context_end_token
            while token_end_index >= context_start_token and offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

2.5 Apply Preprocessing (Optimized for Speed)

In [7]:
tokenized_datasets = dataset.map(preprocess_function, batched=True)

#  Speed optimization
train_dataset = tokenized_datasets["train"].select(range(10000))
valid_dataset = tokenized_datasets["validation"].select(range(2000))

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

2.6 Fine-Tuning the Model

In [9]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="qa_finetuned",
    save_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=200,
    report_to="none",
    fp16=True if __import__("torch").cuda.is_available() else False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
)

trainer.train()

Step,Training Loss
200,3.311210
400,1.894229
600,1.611608
800,1.379748
1000,1.355876
1200,1.287162


TrainOutput(global_step=1250, training_loss=1.783411752319336, metrics={'train_runtime': 223.8913, 'train_samples_per_second': 44.665, 'train_steps_per_second': 5.583, 'total_flos': 1959725675520000.0, 'train_loss': 1.783411752319336, 'epoch': 1.0})

2.7 Evaluation (EM & F1)

In [10]:
import evaluate
import numpy as np

metric = evaluate.load("squad")

pred = trainer.predict(valid_dataset)
start_logits, end_logits = pred.predictions

raw_valid = dataset["validation"].select(range(2000))

predictions = []
references = []

for i in range(len(start_logits)):
    start = np.argmax(start_logits[i])
    end = np.argmax(end_logits[i]) + 1

    offsets = valid_dataset[i]["offset_mapping"]
    context = raw_valid[i]["context"]

    if start < len(offsets) and end <= len(offsets):
        start_char = offsets[start][0]
        end_char = offsets[end - 1][1]
        answer = context[start_char:end_char]
    else:
        answer = ""

    predictions.append({"id": raw_valid[i]["id"], "prediction_text": answer})
    references.append({"id": raw_valid[i]["id"], "answers": raw_valid[i]["answers"]})

score = metric.compute(predictions=predictions, references=references)

print(score)

{'exact_match': 63.55, 'f1': 72.10013892343385}


2.8 Custom Testing

In [12]:
import torch

def get_answer(question, context):
    inputs = tokenizer(question, context, return_tensors="pt")

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1

    return tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start:end])
    )

# Example 1
print(get_answer(
    "Who developed the theory of relativity?",
    "Albert Einstein developed the theory of relativity."
))

# Example 2
print(get_answer(
    "Where is the Eiffel Tower located?",
    "The Eiffel Tower is located in Paris, France."
))

albert einstein
paris, france


## 3. Results


The fine-tuned model achieved the following performance on the validation dataset:

Exact Match (EM): 63.55%
F1 Score: 72.10%
Custom Test Outputs:
Q1: Who developed the theory of relativity?
→ Albert Einstein
Q2: Where is the Eiffel Tower located?
→ Paris, France

The higher F1 score compared to EM indicates that the model often predicts partially correct answers even when the exact span is not perfectly matched.

## 4. Reflection

Fine-tuning a transformer model for question answering provided valuable insights into how modern NLP systems operate beyond traditional classification tasks. Unlike classification, QA requires predicting exact spans within a context, which makes preprocessing more complex. One of the main challenges was correctly aligning answer spans with tokenized inputs. I also learned how powerful pre-trained models like BERT can be when adapted to specific tasks. The Hugging Face ecosystem greatly simplified model training and evaluation. Overall, this assignment enhanced my understanding of transformer-based architectures and their real-world applications.

# **Thank You**

**Forkan Amin**

*AI Engineering*